# Description du projet

Ce projet s’appuie sur le jeu de données **Diabetes 130-US hospitals for years 1999–2008** publié par l’**UCI Machine Learning Repository**.  

L’objectif est de **prédire la probabilité de réadmission d’un patient diabétique dans les 30 jours** suivant une hospitalisation, à partir d’informations **démographiques, médicales et administratives**.

---

## Objectifs

- Comprendre les **facteurs associés à la réadmission** hospitalière des patients diabétiques.  
- Construire un **modèle prédictif** robuste et interprétable.  
- Comparer différentes **méthodes de sélection de variables** et de **modélisation**.  

---

## Étapes du projet

### 1. Nettoyage et prétraitement

- Remplacement des valeurs manquantes `"?"` par `NaN`, puis suppression ou imputation.  
- Suppression des colonnes inutilisables ou quasi vides (`weight`, `payer_code`, `examide`, etc.).  
- Encodage ordinal des variables liées aux médicaments (`"No" → 0`, `"Steady" → 1`, etc.).  
- Définition des groupes de variables :
  - `num_cols` : variables numériques, standardisées avec `StandardScaler`.  
  - `ohe_cols` : variables catégorielles, encodées via `OneHotEncoder`.  
  - `other_cols` : variables déjà numériques.  
- Construction d’un **pipeline scikit-learn** (via `ColumnTransformer`).  
- Création de la cible binaire :  
  - `y = 1` si `readmitted == "<30"`  
  - `y = 0` sinon.

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
%pip install ucimlrepo

In [ ]:
# importation de la base depuis UCI 
from ucimlrepo import fetch_ucirepo 

diabetes_130_us_hospitals_for_years_1999_2008 = fetch_ucirepo(id=296) 

X = diabetes_130_us_hospitals_for_years_1999_2008.data.features 
Y = diabetes_130_us_hospitals_for_years_1999_2008.data.targets 

# variable information 
print(diabetes_130_us_hospitals_for_years_1999_2008.variables) 


In [ ]:
# Fusion X + Y
df = pd.concat([X, Y], axis=1)
print("Fusion terminée : ", df.shape)
# Affichage des premières lignes
df.head()

In [ ]:
num_cols  = df.select_dtypes(include=["number"]).columns.tolist()
cat_cols  = df.select_dtypes(include=["object"]).columns.tolist()

print(num_cols)
print(cat_cols)          


In [ ]:
# A1Cresult / max_glu_serum sont catégorielles cliniques ; on garde en cat
for c in ["A1Cresult", "max_glu_serum"]:
    if c in df.columns and c not in cat_cols:
        cat_cols.append(c)
        if c in num_cols:
            num_cols.remove(c)
# Colonnes à supprimer (avec au moins 40% de valeurs manquantes)
cols_to_remove = ["payer_code", "medical_specialty", "max_glu_serum", "A1Cresult", "weight"]

X = df.drop(columns=cols_to_remove)

# Vérification
print(f"Colonnes supprimées : {cols_to_remove}")

In [ ]:
# Remplacer les mentions textuelles par de vrais NaN
X = X.replace(["missing value", "Missing value", "Missing Value"], np.nan)

initial_rows = X.shape[0]

# Supprimer toutes les lignes contenant au moins un NaN
X = X.dropna()

# Nombre de lignes après suppression
final_rows = X.shape[0]

# Pourcentage de lignes supprimées
pct_removed = 100 * (initial_rows - final_rows) / initial_rows

# Vérification
print(f"Lignes restantes après suppression : {final_rows}")
print(f"Pourcentage de lignes supprimées : {pct_removed:.2f}%")

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# 1) Encodage ordinal des colonnes de médicaments

drug_cols = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide",
    "glimepiride", "acetohexamide", "glipizide", "glyburide", "tolbutamide",
    "pioglitazone", "rosiglitazone", "acarbose", "miglitol", "troglitazone",
    "tolazamide", "examide", "citoglipton", "insulin", "glyburide-metformin",
    "glipizide-metformin", "glimepiride-pioglitazone", "metformin-rosiglitazone",
    "metformin-pioglitazone", "change", "diabetesMed"
]

# Classification ordinale des médicaments
drug_map = {"No": 0, "Steady": 1, "Up": 2, "Down": -1}


for col in drug_cols:
    if col in X.columns:
        X[col] = X[col].map(drug_map).fillna(0).astype(int)

print("Médicaments encodés (mapping ordinal appliqué).")

In [ ]:
# 2) Définir les groupes de variables

num_cols = [
    "admission_type_id","discharge_disposition_id","admission_source_id",
    "time_in_hospital","num_lab_procedures","num_procedures",
    "num_medications","number_outpatient","number_emergency",
    "number_inpatient","number_diagnoses"
]
num_cols = [c for c in num_cols if c in X.columns]  # sécurité

ohe_cols = ["race","gender","age"]
ohe_cols = [c for c in ohe_cols if c in X.columns]  # sécurité

# Les colonnes restantes (autres features déjà numériques)
other_cols = [c for c in X.columns if c not in num_cols + ohe_cols]

print(f"Numériques standardisées : {num_cols}")
print(f"Catégorielles à One-Hot : {ohe_cols}")
print(f"Déjà numériques (médicaments encodés) : {len(other_cols)} colonnes")

In [ ]:
# 3) Pipelines

numeric_tf = Pipeline(steps=[
    ("scale", StandardScaler())
])

categorical_tf = Pipeline(steps=[
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))
])


# ColumnTransformer
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_tf, num_cols),
        ("ohe", categorical_tf, ohe_cols),
        ("pass", "passthrough", other_cols)  # on garde les autres telles quelles
    ]
)

In [ ]:

# 5) Fit-transform & reconstruction DataFrame

X_mat = preprocess.fit_transform(X)

# Récupération des noms des colonnes encodées
ohe = preprocess.named_transformers_["ohe"].named_steps["onehot"]
ohe_feature_names = ohe.get_feature_names_out(ohe_cols)
feature_names = list(num_cols) + list(ohe_feature_names) + other_cols

X_final = pd.DataFrame(X_mat, columns=feature_names, index=X.index)

print("Shape finale :", X_final.shape)
X_final.head()

# Liste simple
print(X_final.columns.tolist())

In [ ]:
# Définir la cible
y = X_final["readmitted"].copy()

# Transformer en binaire : 1 si réadmission <30 jours, 0 sinon
y = y.apply(lambda x: 1 if x == "<30" else 0)

# Supprimer la colonne cible du dataset
X_final= X_final.drop(columns=["readmitted"])

print(f"X shape: {X_final.shape}")
print(f"Y shape: {y.shape}")
print(f"Valeurs uniques de y : {y.unique()}")


In [ ]:
# Crée le dossier data
os.makedirs("data", exist_ok=True)

# Sauvegarde X_final et y
X_final.to_csv("data/X_final.csv", index=False)
y.to_csv("data/y.csv", index=False)

### 2. Sélection de variables

#### 2.1. Nombre optimal de variables (cross-validation)

- Utilisation d’une **régression logistique** avec **Stratified K-Fold Cross-Validation**.  
- Pour différents nombres de variables `k` :
  - Sélection des `k` meilleures variables selon un critère donné.  
  - Évaluation de l’**AUC** moyen.  
- Choix du **plus petit `k`** donnant la **meilleure performance moyenne**, afin de concilier :
  - **Simplicité** du modèle,
  - **Performance prédictive**,
  - Limitation de la **sur-sélection**.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFECV
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split, GridSearchCV
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from matplotlib_venn import venn3
import os


In [ ]:
# Charger les données
X = pd.read_csv("data/X_final.csv")
y = pd.read_csv("data/y.csv").squeeze()  # transforme en série

print("Dimensions :", X.shape, y.shape)

In [ ]:
# Identifier les colonnes constantes
constant_cols = [col for col in X.columns if X[col].nunique() <= 1]

# Affichage des colonnes concernées
if constant_cols:
    print(f"{len(constant_cols)} variable(s) constante(s) détectée(s) :")
    print(constant_cols)
    
    # Suppression
    X = X.drop(columns=constant_cols)
    print(f"Nouveau nombre de variables : {X.shape[1]}")
else:
    print("Aucune variable constante détectée.")


In [ ]:
# diag_1, diag_2, diag_3 sont des codes ICD-9. On les regroupe en catégories médicales.

def map_icd9(code):
    if code.startswith('V') or code.startswith('E'):
        return 'Other'
    try:
        c = float(code)
    except:
        return 'Other' 
    if 390 <= c <= 459 or c == 785: return 'Circulatory'
    if 460 <= c <= 519 or c == 786: return 'Respiratory'
    if 520 <= c <= 579 or c == 787: return 'Digestive'
    if str(int(c)).startswith('250'): return 'Diabetes'
    if 800 <= c <= 999: return 'Injury'
    if 710 <= c <= 739: return 'Musculoskeletal'
    if 580 <= c <= 629 or c == 788: return 'Genitourinary'
    if 140 <= c <= 239: return 'Neoplasms'
    return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    X[col] = X[col].astype(str).apply(map_icd9)


In [ ]:
# Identifie les colonnes non numériques (normalement diag_1, diag_2, diag_3)
non_numeric_cols = X.select_dtypes(exclude=['number']).columns
print("Colonnes non numériques :", list(non_numeric_cols))

# Encodage ordinal
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[non_numeric_cols] = encoder.fit_transform(X[non_numeric_cols])

# Vérification
print(X.dtypes.value_counts())


In [ ]:
# Détermination du nombre optimal de variables par cross-validation avec RFECV

# 1) Train / Test split (évite l'optimisme)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


# 2) Modèle
logreg = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    class_weight="balanced",
    max_iter=3000
)

# 3) RFECV → sélection multivariée + détermination automatique de k
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rfecv = RFECV(
    estimator=logreg,
    step=1,                 
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

# 4) Pipeline
pipeline = Pipeline([
    ("rfecv", rfecv)
])

pipeline.fit(X_train, y_train)

# 5) Résultats : nombre optimal de variables
optimal_k = pipeline.named_steps["rfecv"].n_features_
support = pipeline.named_steps["rfecv"].support_
ranking = pipeline.named_steps["rfecv"].ranking_

print("✔ Nombre optimal de variables k :", optimal_k)
print("✔ Variables sélectionnées (support mask) :", support)

# 7) Graphique performance (AUC) en fonction du nombre de features
results = rfecv.cv_results_
mean_scores = results["mean_test_score"]

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(mean_scores) + 1), mean_scores, marker='o')
plt.axvline(optimal_k, color='red', linestyle='--', label=f"k optimal = {optimal_k}")
plt.xlabel("Nombre de variables")
plt.ylabel("AUC (validation croisée)")
plt.title("RFECV – Performance en fonction du nombre de variables")
plt.grid(True)
plt.legend()
plt.show()


# 8) Score Final TEST
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba)

print("✔ AUC TEST :", test_auc)


#### 2.2. Méthode 1 — Information Mutuelle (IM)

- Mesure la **dépendance statistique** entre une variable explicative `X` et la cible `Y`.  
- Capte des **relations linéaires et non linéaires**, contrairement à la corrélation de Pearson.  
- Méthode de type **filtre** : chaque variable est évaluée indépendamment du modèle → rapide, générique.

In [ ]:
# Sélection selon l'information mutuelle
selector = SelectKBest(score_func=mutual_info_classif, k='all')
X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]
scores = selector.scores_[selector.get_support()]

# Affichage
feat_scores = pd.DataFrame({'Variable': selected_features, 'Score': scores})
feat_scores.sort_values('Score', ascending=False, inplace=True)
print(feat_scores)


In [ ]:
# Tri des variables par score décroissant
feat_scores_sorted = feat_scores.sort_values('Score', ascending=False).reset_index(drop=True)

# TOP 25 variables
top_k = 25
feat_scores_sorted['Couleur'] = [
    "#ffef0eff" if i < top_k else '#1f77b4'
    for i in range(len(feat_scores_sorted))
]

# Visualisation 
plt.figure(figsize=(10, 8))
plt.barh(
    feat_scores_sorted['Variable'],
    feat_scores_sorted['Score'],
    color=feat_scores_sorted['Couleur'],
    edgecolor='black'
)

plt.title("Importance des variables selon l'information mutuelle", fontsize=13, weight='bold')
plt.xlabel("Score d'information mutuelle")
plt.ylabel("Variables explicatives")
plt.gca().invert_yaxis()

# Valeurs sur les barres 
for i, v in enumerate(feat_scores_sorted['Score']):
    plt.text(v + 0.0005, i, f"{v:.3f}", va='center', fontsize=8)

# Légende simple
plt.legend(
    handles=[
        plt.Rectangle((0, 0), 1, 1, color='#ffef0eff', label='Top 25 variables'),
        plt.Rectangle((0, 0), 1, 1, color='#1f77b4', label='Autres variables')
    ],
    loc='best'
)

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



#### 2.3. Méthode 2 — Régression Lasso (L1)

- Régression logistique pénalisée **L1** (Lasso) → méthode **embedded**.  
- Ajout d’une pénalisation `λ Σ |βᵢ|` : certains coefficients deviennent **nuls**, ce qui **élimine automatiquement** les variables peu utiles.  
- Avantages :
  - Sélection automatique des variables les plus explicatives,  
  - Réduction du **sur-apprentissage**,  
  - Modèle linéaire **interprétable**.

In [ ]:
# LASSO Logistic Regression avec sélection des variables (version optimisée)

# 1) TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape :", X_train.shape)
print("Test shape :", X_test.shape)

# 1bis) SOUS-ÉCHANTILLON POUR LA CV (20k lignes = idéal)
X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train,
    train_size=20000,
    stratify=y_train,
    random_state=42
)

print("Tune shape :", X_tune.shape)

# 2) CV RAPIDE POUR TROUVER LE MEILLEUR C
log_reg_cv = LogisticRegressionCV(
    Cs=np.logspace(-3, 2, 10),
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    penalty='l1',
    solver='liblinear',
    scoring='roc_auc',
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    n_jobs=-1,
    refit=True
)

log_reg_cv.fit(X_tune, y_tune)

best_C = log_reg_cv.C_[0]
print("\nMeilleur C sélectionné :", best_C)

# 3) MODELE LASSO FINAL SUR TOUT LE TRAIN
final_lasso = LogisticRegression(
    C=best_C,
    penalty='l1',
    solver='liblinear',
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)

final_lasso.fit(X_train, y_train)

# 4) EXTRACTION DES COEFFS
coef = final_lasso.coef_[0]
variables = X.columns

lasso_df = pd.DataFrame({
    'Variable': variables,
    'Coefficient': coef,
    'Abs_Coefficient': np.abs(coef),
    'Selected': coef != 0
})

# Trier par importance
lasso_df_sorted = lasso_df.sort_values('Abs_Coefficient', ascending=False)

# Garder les variables sélectionnées
selected_vars = lasso_df_sorted[lasso_df_sorted['Selected']]

print("\nVariables sélectionnées :", selected_vars.shape[0])
display(selected_vars)

# 5) PLOT
plt.figure(figsize=(10, max(6, len(selected_vars)*0.3)))
plt.barh(
    selected_vars['Variable'],
    selected_vars['Abs_Coefficient'],
    color="#ffbf00",
    edgecolor="black"
)
plt.xlabel("Coefficient absolu (importance)")
plt.title("Variables sélectionnées par LASSO (coefficients ≠ 0)")
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


#### 2.4. Méthode 3 — XGBoost

- Modèle d’**arbres de décision boostés** (gradient boosting).  
- Importance des variables basée sur :
  - la fréquence d’utilisation dans les arbres,  
  - le **gain** de réduction d’erreur,  
  - la **couverture** des observations.  
- Avantages :
  - Capture des **interactions non linéaires**,  
  - Robustesse aux données bruitées,  
  - Excellentes performances empiriques.

In [ ]:

# importance des variables avec XGBoost (gain)

# 0) Nettoyage des noms de colonnes AVANT le split
X = X.copy()
X.columns = X.columns.astype(str)
X.columns = (
    X.columns.str.replace('[', '(', regex=False)
              .str.replace(']', ')', regex=False)
              .str.replace('<', 'inf', regex=False)
              .str.replace('>', 'sup', regex=False)
)

# 1) TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)



print("Train shape :", X_train.shape)
print("Test shape :", X_test.shape)

# 2) GRIDSEARCHCV (optimisation des hyperparamètres)

param_grid = {
    "n_estimators": [150, 250],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5, 7],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

xgb_base = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1
)




grid.fit(X_train, y_train)

print("✔ Meilleurs hyperparamètres :", grid.best_params_)
print("✔ Meilleur score CV AUC :", grid.best_score_)


# 3) ENTRAÎNEMENT FINAL AVEC LES MEILLEURS PARAMÈTRES

best_xgb = grid.best_estimator_
best_xgb.fit(X_train, y_train)


# 4) IMPORTANCE DES VARIABLES BASÉE SUR LE GAIN

booster = best_xgb.get_booster()
gain_dict = booster.get_score(importance_type='gain')

xgb_gain = pd.DataFrame({
    "Variable": gain_dict.keys(),
    "Importance_gain": gain_dict.values()
})

# Mettre 0 pour les variables jamais utilisées
all_vars = pd.DataFrame({"Variable": X_train.columns})
xgb_gain = all_vars.merge(xgb_gain, on="Variable", how="left").fillna(0)

# Tri décroissant
xgb_gain = xgb_gain.sort_values("Importance_gain", ascending=False).reset_index(drop=True)


# 5) ON GARDE k=25 
top_k = 25
xgb_gain["Couleur"] = ["#ff7f0e" if i < top_k else "#1f77b4" 
                       for i in range(len(xgb_gain))]


# 6) VISUALISATION 

plt.figure(figsize=(10, 8))
plt.barh(
    xgb_gain["Variable"],
    xgb_gain["Importance_gain"],
    color=xgb_gain["Couleur"],
    edgecolor="black"
)
plt.gca().invert_yaxis()

plt.title("Importance des variables XGBoost (GAIN)", fontsize=13, weight="bold")
plt.xlabel("Gain moyen par split")
plt.ylabel("Variables")

plt.legend(
    handles=[
        plt.Rectangle((0, 0), 1, 1, color="#ff7f0e", label=f"Top {top_k} variables"),
        plt.Rectangle((0, 0), 1, 1, color="#1f77b4", label="Autres variables")
    ],
    loc="best"
)

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 7) PERFORMANCE FINALE SUR LE TEST

y_pred_proba = best_xgb.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba)

print("✔ AUC TEST :", round(test_auc, 4))


#### 2.5. Agrégation des scores

- Chaque méthode fournit une vision complémentaire :  
  - **IM** : dépendance statistique,  
  - **Lasso** : pouvoir discriminant linéaire,  
  - **XGBoost** : effets non linéaires et interactions.  
- Normalisation des scores entre 0 et 1, puis combinaison en un **score global** :

\[
\text{Score\_global} = 0.4 \times \text{IM} + 0.3 \times \text{Lasso} + 0.3 \times \text{XGBoost}
\]

- Les variables avec le **score global le plus élevé** sont retenues pour la base finale du modèle.

---

In [ ]:
# --- INFORMATION MUTUELLE ---
mi_rank = feat_scores[['Variable', 'Score']].copy()
mi_rank.rename(columns={'Score': 'MI_Score'}, inplace=True)

# --- LASSO ---
lasso_rank = lasso_df_sorted[['Variable', 'Abs_Coefficient']].copy()
lasso_rank.rename(columns={'Abs_Coefficient': 'LASSO_Score'}, inplace=True)


# --- XGBOOST ---
xgb_rank = xgb_gain[['Variable', 'Importance_gain']].copy()
xgb_rank.rename(columns={'Importance_gain': 'XGB_Gain'}, inplace=True)

# --- MERGE ---
merged = (
    mi_rank.merge(lasso_rank, on='Variable', how='outer')
           .merge(xgb_rank, on='Variable', how='outer')
)

# Remplacer NaN par 0 (si une méthode ne sélectionne pas une variable)
merged.fillna(0, inplace=True)

display(merged)


In [ ]:
# --- 1. Calcul des rangs individuels ---
merged['MI_rank'] = merged['MI_Score'].rank(ascending=False, method='min')
merged['LASSO_rank'] = merged['LASSO_Score'].rank(ascending=False, method='min')
merged['XGB_rank'] = merged['XGB_Gain'].rank(ascending=False, method='min')

# --- 2. Score de Borda ---
N = len(merged)

merged['MI_borda'] = N - merged['MI_rank'] + 1
merged['LASSO_borda'] = N - merged['LASSO_rank'] + 1
merged['XGB_borda'] = N - merged['XGB_rank'] + 1

merged['Borda_total'] = (
    merged['MI_borda'] +
    merged['LASSO_borda'] +
    merged['XGB_borda']
)

# --- 3. Moyenne des rangs ---
merged['Mean_rank'] = (
    merged[['MI_rank', 'LASSO_rank', 'XGB_rank']]
    .mean(axis=1)
)

# --- 4. Classement final ---
final_ranking = merged.sort_values([
    'Mean_rank',        # classement principal
    'Borda_total'       # critère secondaire
], ascending=[True, False])

display(final_ranking)


In [ ]:
import seaborn as sns

# Sélection des colonnes à afficher
heatmap_data = merged[['Variable', 'MI_rank', 'LASSO_rank', 'XGB_rank']].copy()
heatmap_data.set_index('Variable', inplace=True)

plt.figure(figsize=(10, max(6, len(heatmap_data)*0.25)))
sns.heatmap(heatmap_data, cmap="viridis_r", annot=True, fmt=".0f")
plt.title("Heatmap des rankings MI / LASSO / XGBoost")
plt.ylabel("Variables")
plt.xlabel("Méthodes")
plt.tight_layout()
plt.show()


In [ ]:
top_combined = final_ranking.head(25).copy()

plt.figure(figsize=(10, 8))
plt.barh(
    top_combined['Variable'],
    top_combined['Mean_rank'],  
    color="#ff7f0e",
    edgecolor="black"
)

plt.title("Top 25 variables combinées (Mean Rank)", fontsize=14, weight="bold")
plt.xlabel("Importance (Mean Rank)")
plt.ylabel("Variables")
plt.gca().invert_yaxis()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:


#  Définition des ensembles 
mi_set = set(mi_rank.sort_values('MI_Score', ascending=False).head(25)['Variable'])
lasso_set = set(lasso_rank[lasso_rank['LASSO_Score'] > 0]['Variable'])
xgb_set = set(xgb_rank.sort_values('XGB_Gain', ascending=False).head(25)['Variable'])


# 1) TABLEAU D'APPARTENANCE DES VARIABLES 


# Ensemble total des variables présentes dans au moins un set
all_vars = mi_set | lasso_set | xgb_set

# Construction du tableau 
membership_df = pd.DataFrame({
    "Variable": list(all_vars),
    "In_MI": [var in mi_set for var in all_vars],
    "In_LASSO": [var in lasso_set for var in all_vars],
    "In_XGB": [var in xgb_set for var in all_vars]
})

# Ajouter une colonne : nombre de méthodes où la variable apparaît
membership_df["Nb_oui"] = membership_df[["In_MI", "In_LASSO", "In_XGB"]].sum(axis=1)

# Trier par Nb_oui décroissant
membership_df = membership_df.sort_values(by="Nb_oui", ascending=False)

# Affichage final
print("\n Tableau d'appartenance trié (plus de 'Oui' en haut) :")
display(membership_df[["Variable", "In_MI", "In_LASSO", "In_XGB", "Nb_oui"]])

# 2) VENN DIAGRAM

plt.figure(figsize=(8, 6))
venn3([mi_set, lasso_set, xgb_set],
      ('Information Mutuelle', 'LASSO', 'XGBoost (Gain)'))
plt.title("Venn Diagram des variables sélectionnées", fontsize=14)
plt.show()




In [ ]:
# 1) Récupération et correction des noms
top25_vars = (
    final_ranking.head(25)["Variable"]
    .str.replace("[", "(", regex=False)
    .str.replace("]", ")", regex=False)
    .tolist()
)

# 2) Filtrage : on garde uniquement les colonnes qui existent dans X
top25_vars = [v for v in top25_vars if v in X.columns]

# 3) Extraction dans X
top25_dataset = X[top25_vars].copy()

# 4) Enregistrement
output_path = os.path.join("data", "X_selections.csv")
top25_dataset.to_csv(output_path, index=False, encoding="utf-8")

print(f"✅ Base top25 enregistrée dans : {output_path}")
print(f"📂 Taille de la base : {top25_dataset.shape}")


### 3. Modélisation et évaluation

#### 3.1. Méthodologie d’évaluation

- Cible **fortement déséquilibrée** (~11 % de réadmissions < 30 jours).  
- Métrique principale : **PR-AUC** (Precision–Recall AUC), plus adaptée que la ROC-AUC en cas de déséquilibre.  

Métriques utilisées :

- **PR-AUC** : critère principal.  
- **ROC-AUC** : performance globale de classement.  
- **F1-score** : compromis précision / rappel.  
- **Brier Score** : qualité de la **calibration** des probabilités.  
- **Recall@Top 20 %** : part de patients réadmis dans les 20 % jugés les plus à risque.

- Split des données : **80 % train / 20 % test**, stratifié.  
- Tous les prétraitements et la sélection de variables sont intégrés au **pipeline**, pour éviter toute **fuite de données**.

#### 3.2. Modèle interprétable — Régression Logistique L1

- Le modèle final retenu (dans une première approche interprétable) est la **régression logistique L1**.  
- Il combine :
  - **Interprétabilité** (coefficients lisibles),  
  - **Sélection automatique** des variables,  
  - **Robustesse** grâce à la pénalisation L1.

In [ ]:

# === 0. IMPORTS ===
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    roc_curve, classification_report, confusion_matrix, brier_score_loss

)


from sklearn.linear_model import LogisticRegressionCV
from sklearn.calibration import CalibratedClassifierCV

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# === 1. CHARGEMENT DES DONNÉES ===

X = pd.read_csv("data/X_selections.csv")
y = pd.read_csv("data/y.csv").squeeze()
y.name = "target"

print(f"Shape X: {X.shape} | Shape y: {y.shape}")
print("Distribution de la cible (train/test avant split):\n",
      y.value_counts(normalize=True).round(3))

# Split stratifié train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

In [ ]:
# === 2. MODELE LOGISTIQUE  ===

logit = LogisticRegression(
             
    solver="liblinear",
    class_weight="balanced",
    max_iter=2000,
    random_state=RANDOM_STATE
)

logit.fit(X_train, y_train)

In [ ]:
# === 4. ÉVALUATION SUR TEST ===

proba_test = logit.predict_proba(X_test)[:, 1]

pr_auc = average_precision_score(y_test, proba_test)
roc_auc = roc_auc_score(y_test, proba_test)
brier = brier_score_loss(y_test, proba_test)

print(f"\n=== Performances sur test ===")
print(f"PR-AUC : {pr_auc:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Brier  : {brier:.5f}")

In [ ]:
# === 5. métrique ===

prec, rec, thr = precision_recall_curve(y_test, proba_test)
f1 = 2 * prec * rec / (prec + rec + 1e-12)
best_idx = int(np.nanargmax(f1))
best_thr = thr[min(max(best_idx-1, 0), len(thr)-1)] if len(thr) > 0 else 0.5

print(f"\nSeuil optimal (F1 max): {best_thr:.4f}")

y_pred = (proba_test >= best_thr).astype(int)

print("\n=== Rapport de classification (test) ===")
print(classification_report(y_test, y_pred, digits=3))

cm = confusion_matrix(y_test, y_pred)
print("Matrice de confusion (test):\n", cm)

In [ ]:
# === 6. INTERPRÉTATION DES COEFFICIENTS ===

coef = pd.Series(logit.coef_.ravel(), index=X_train.columns, name="beta")
odds = np.exp(coef).rename("odds_ratio")

interpret_table = pd.concat([coef, odds], axis=1)
interpret_table["abs_beta"] = interpret_table["beta"].abs()
interpret_table = interpret_table.sort_values("abs_beta", ascending=False)

print("\nTop 20 variables par |β| :")
print(interpret_table[["beta", "odds_ratio"]].head(20))

# Sauvegarde
Path("outputs").mkdir(exist_ok=True)
interpret_table.to_csv("outputs/logreg_interpretation.csv")
pd.DataFrame({"y_true": y_test.values,
              "proba": proba_test,
              "y_pred": y_pred}).to_csv(
    "outputs/test_predictions.csv", index=False
)

In [ ]:
# === 7. VISUALISATIONS ===

# Calibration
plt.figure(figsize=(6,4))
CalibrationDisplay.from_predictions(y_test, proba_test, n_bins=10)
plt.title("Courbe de calibration (isotonic)")
plt.tight_layout()
plt.show()

# PR Curve
plt.figure(figsize=(6,4))
plt.plot(rec, prec, label=f"PR-AUC={pr_auc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall (test)")
plt.legend()
plt.tight_layout()
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, proba_test)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f"ROC-AUC={roc_auc:.3f}")
plt.plot([0,1], [0,1], linestyle='--', alpha=0.5)
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC (test)")
plt.legend()
plt.tight_layout()
plt.show()

# Histogramme
plt.figure(figsize=(6,4))
plt.hist(proba_test, bins=30)
plt.xlabel("Probabilité prédite")
plt.ylabel("Comptes")
plt.title("Distribution des probabilités (test)")
plt.tight_layout()
plt.show()


In [ ]:
# === 2bis. MODELE LOGISTIQUE AVEC SMOTE ===
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

smote_logit = ImbPipeline(steps=[
    ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        solver="liblinear",
        class_weight="balanced",   # tu peux mettre None si tu veux éviter double correction
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

smote_logit.fit(X_train, y_train)

# Prédictions SMOTE
proba_test_sm = smote_logit.predict_proba(X_test)[:, 1]

# Scores SMOTE
pr_auc_sm = average_precision_score(y_test, proba_test_sm)
roc_auc_sm = roc_auc_score(y_test, proba_test_sm)
brier_sm = brier_score_loss(y_test, proba_test_sm)

print("\n=== Performances SMOTE (test) ===")
print(f"PR-AUC : {pr_auc_sm:.4f}")
print(f"ROC-AUC: {roc_auc_sm:.4f}")
print(f"Brier  : {brier_sm:.5f}")

# Seuil optimal SMOTE
prec_sm, rec_sm, thr_sm = precision_recall_curve(y_test, proba_test_sm)
f1_sm = 2 * prec_sm * rec_sm / (prec_sm + rec_sm + 1e-12)
best_idx_sm = int(np.nanargmax(f1_sm))
best_thr_sm = thr_sm[min(max(best_idx_sm-1, 0), len(thr_sm)-1)] if len(thr_sm) > 0 else 0.5

print(f"\nSeuil optimal SMOTE (F1 max): {best_thr_sm:.4f}")

y_pred_sm = (proba_test_sm >= best_thr_sm).astype(int)

print("\n=== Rapport de classification SMOTE (test) ===")
print(classification_report(y_test, y_pred_sm, digits=3))


In [ ]:
print("\n=== COMPARAISON LOGIT vs LOGIT+SMOTE ===")

df_comp = pd.DataFrame({
    "Metric": ["ROC-AUC", "PR-AUC", "Brier"],
    "Logit": [roc_auc, pr_auc, brier],
    "SMOTE+Logit": [roc_auc_sm, pr_auc_sm, brier_sm]
})

print(df_comp)


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, f1_score
import numpy as np

# === MODELE DE BASE ===
dt = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)

# === PETITE GRILLE OPTIMISÉE (rapide + efficace) ===
dt_params = {
    "max_depth": [3, 5, 7, 10, None],      
    "min_samples_leaf": [20, 50, 100],    
    "min_samples_split": [2, 5, 10]       
}

dt_cv = GridSearchCV(
    estimator=dt,
    param_grid=dt_params,
    scoring="average_precision",   # PR-AUC = dynamique pour classes déséquilibrées
    cv=5,
    n_jobs=-1,
    verbose=0
)

# Entraînement
dt_cv.fit(X_train, y_train)

print("Meilleurs paramètres (Decision Tree) :")
print(dt_cv.best_params_)

dt_model = dt_cv.best_estimator_


In [ ]:
# === 4bis. ÉVALUATION ARBRE DE DÉCISION SUR TEST ===

proba_test_tree = dt_model.predict_proba(X_test)[:, 1]

pr_auc_tree = average_precision_score(y_test, proba_test_tree)
roc_auc_tree = roc_auc_score(y_test, proba_test_tree)
brier_tree = brier_score_loss(y_test, proba_test_tree)

print("\n=== Performances ARBRE sur test ===")
print(f"PR-AUC : {pr_auc_tree:.4f}")
print(f"ROC-AUC: {roc_auc_tree:.4f}")
print(f"Brier  : {brier_tree:.5f}")

# Seuil optimal comme pour le logit : F1 max
prec_tree, rec_tree, thr_tree = precision_recall_curve(y_test, proba_test_tree)
f1_tree = 2 * prec_tree * rec_tree / (prec_tree + rec_tree + 1e-12)
best_idx_tree = int(np.nanargmax(f1_tree))
best_thr_tree = thr_tree[min(max(best_idx_tree-1, 0), len(thr_tree)-1)] if len(thr_tree) > 0 else 0.5

print(f"\nSeuil optimal ARBRE (F1 max): {best_thr_tree:.4f}")

y_pred_tree = (proba_test_tree >= best_thr_tree).astype(int)

print("\n=== Rapport de classification ARBRE (test) ===")
print(classification_report(y_test, y_pred_tree, digits=3))

cm_tree = confusion_matrix(y_test, y_pred_tree)
print("Matrice de confusion ARBRE (test):\n", cm_tree)


In [ ]:
feat_imp = pd.Series(dt_model.feature_importances_, index=X_train.columns)
feat_imp = feat_imp.sort_values(ascending=False)

plt.figure(figsize=(6,10))
feat_imp.head(20).plot(kind="barh")
plt.title("Top 20 Features – Decision Tree")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# === 9bis. COMPARAISON LOGIT vs SMOTE+LOGIT vs ARBRE ===

df_comp = pd.DataFrame({
    "Metric": ["ROC-AUC", "PR-AUC", "Brier"],
    "Logit": [roc_auc, pr_auc, brier],
    "SMOTE+Logit": [roc_auc_sm, pr_auc_sm, brier_sm],
    "DecisionTree": [roc_auc_tree, pr_auc_tree, brier_tree]
})

print("\n=== COMPARAISON DES MODELES ===")
print(df_comp)


In [ ]:
# === PLOTS (ROC & PR) POUR L’ARBRE DE DÉCISION ===

from sklearn.metrics import roc_curve, precision_recall_curve

# Courbes
fpr_tree, tpr_tree, _ = roc_curve(y_test, proba_test_tree)
prec_tree, rec_tree, _ = precision_recall_curve(y_test, proba_test_tree)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# ----------------------------------------------
# 1) ROC Curve
# ----------------------------------------------
ax[0].plot(fpr_tree, tpr_tree, label=f"Arbre (AUC = {roc_auc_tree:.3f})")
ax[0].plot([0, 1], [0, 1], "--", color="grey")
ax[0].set_title("ROC Curve - Arbre de Décision", fontsize=14)
ax[0].set_xlabel("False Positive Rate")
ax[0].set_ylabel("True Positive Rate")
ax[0].grid(True)
ax[0].legend()

# ----------------------------------------------
# 2) Precision–Recall Curve
# ----------------------------------------------
ax[1].plot(rec_tree, prec_tree, label=f"Arbre (PR-AUC = {pr_auc_tree:.3f})")
ax[1].set_title("Precision–Recall Curve - Arbre de Décision", fontsize=14)
ax[1].set_xlabel("Recall")
ax[1].set_ylabel("Precision")
ax[1].grid(True)
ax[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# === COURBES COMPARATIVES LOGIT vs ARBRE ===

# Courbes logit (déjà calculées normalement)
fpr_log, tpr_log, _ = roc_curve(y_test, proba_test)
prec_log, rec_log, _ = precision_recall_curve(y_test, proba_test)

# Courbes arbre
fpr_tree, tpr_tree, _ = roc_curve(y_test, proba_test_tree)
prec_tree, rec_tree, _ = precision_recall_curve(y_test, proba_test_tree)

fig, ax = plt.subplots(1, 2, figsize=(14,5))

# -----------------------------------------------------
# 1) ROC CURVE COMPARATIVE
# -----------------------------------------------------
ax[0].plot(fpr_log, tpr_log, label=f"Logit (AUC = {roc_auc:.3f})")
ax[0].plot(fpr_tree, tpr_tree, label=f"Arbre (AUC = {roc_auc_tree:.3f})")
ax[0].plot([0,1],[0,1],"--", color="grey")

ax[0].set_title("ROC Curve – Logit vs Arbre", fontsize=14)
ax[0].set_xlabel("False Positive Rate")
ax[0].set_ylabel("True Positive Rate")
ax[0].grid(True)
ax[0].legend()

# -----------------------------------------------------
# 2) PRECISION–RECALL CURVE COMPARATIVE
# -----------------------------------------------------
ax[1].plot(rec_log, prec_log, label=f"Logit (PR-AUC = {pr_auc:.3f})")
ax[1].plot(rec_tree, prec_tree, label=f"Arbre (PR-AUC = {pr_auc_tree:.3f})")

ax[1].set_title("PR Curve – Logit vs Arbre", fontsize=14)
ax[1].set_xlabel("Recall")
ax[1].set_ylabel("Precision")
ax[1].grid(True)
ax[1].legend()

plt.tight_layout()
plt.show()


## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(
    n_estimators=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Grille optimisée : petite mais suffisante
rf_params = {
    "max_depth": [5, 10, 20, None],
    "min_samples_leaf": [10, 30, 50],
    "max_features": ["sqrt", "log2", None]  # None = toutes les features
}

rf_cv = GridSearchCV(
    estimator=rf,
    param_grid=rf_params,
    scoring="average_precision",   # PR-AUC
    cv=5,
    n_jobs=-1,
    verbose=0
)

rf_cv.fit(X_train, y_train)

print("Meilleurs paramètres (Random Forest) :")
print(rf_cv.best_params_)

rf_model = rf_cv.best_estimator_


In [ ]:
# === 4. ÉVALUATION COMPLETE RANDOM FOREST SUR TEST ===

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_recall_curve, classification_report, confusion_matrix
)

# Probabilités et prédictions
proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Métriques probabilistes
roc_auc_rf = roc_auc_score(y_test, proba_rf)
pr_auc_rf = average_precision_score(y_test, proba_rf)
brier_rf = brier_score_loss(y_test, proba_rf)

print("\n=== Performances RANDOM FOREST sur test ===")
print(f"ROC-AUC : {roc_auc_rf:.4f}")
print(f"PR-AUC  : {pr_auc_rf:.4f}")
print(f"Brier   : {brier_rf:.5f}")

# -------------------------------------------------------------------
# === Seuil optimal basé sur le F1-score ===
# -------------------------------------------------------------------
prec_rf, rec_rf, thr_rf = precision_recall_curve(y_test, proba_rf)
f1_rf_curve = 2 * prec_rf * rec_rf / (prec_rf + rec_rf + 1e-12)

best_idx_rf = int(np.nanargmax(f1_rf_curve))
best_thr_rf = thr_rf[min(max(best_idx_rf-1, 0), len(thr_rf)-1)] if len(thr_rf) > 0 else 0.5

print(f"\nSeuil optimal RF (F1 max): {best_thr_rf:.4f}")

# Prédictions binaires
y_pred_rf = (proba_rf >= best_thr_rf).astype(int)

# F1 pos / F1 neg / Harmonic F1
f1_pos_rf = f1_score(y_test, y_pred_rf)
f1_neg_rf = f1_score(1 - y_test, 1 - y_pred_rf)
harm_rf = (2 * f1_pos_rf * f1_neg_rf) / (f1_pos_rf + f1_neg_rf + 1e-12)

print("\n=== F1 Scores ===")
print(f"F1_POS       : {f1_pos_rf:.4f}")
print(f"F1_NEG       : {f1_neg_rf:.4f}")
print(f"Harmonic F1  : {harm_rf:.4f}")

# -------------------------------------------------------------------
# === Rapport complet de classification ===
# -------------------------------------------------------------------
print("\n=== Rapport de classification RANDOM FOREST (test) ===")
print(classification_report(y_test, y_pred_rf, digits=3))

# Matrice de confusion
cm_rf = confusion_matrix(y_test, y_pred_rf)
print("Matrice de confusion RF (test):\n", cm_rf)


In [ ]:
import pandas as pd

fi_rf = pd.Series(rf_model.feature_importances_, index=X_train.columns)
fi_rf = fi_rf.sort_values(ascending=False)

plt.figure(figsize=(8,12))
fi_rf.head(20).plot(kind="barh")
plt.title("Top 20 Features – Random Forest")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve
import matplotlib.pyplot as plt

# Logit
fpr_log, tpr_log, _ = roc_curve(y_test, proba_test)
prec_log, rec_log, _ = precision_recall_curve(y_test, proba_test)

# Arbre
fpr_dt, tpr_dt, _ = roc_curve(y_test, proba_dt)
prec_dt, rec_dt, _ = precision_recall_curve(y_test, proba_dt)

# Random Forest
fpr_rf, tpr_rf, _ = roc_curve(y_test, proba_rf)
prec_rf, rec_rf, _ = precision_recall_curve(y_test, proba_rf)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves
ax[0].plot(fpr_log, tpr_log, label=f"Logit (AUC={roc_auc:.3f})")
ax[0].plot(fpr_dt, tpr_dt, label=f"Arbre (AUC={roc_dt:.3f})")
ax[0].plot(fpr_rf, tpr_rf, label=f"RF (AUC={roc_rf:.3f})")
ax[0].plot([0,1],[0,1],"--", color="grey")
ax[0].set_title("ROC Curve – Comparaison modèles")
ax[0].set_xlabel("FPR")
ax[0].set_ylabel("TPR")
ax[0].grid(True)
ax[0].legend()

# PR curves
ax[1].plot(rec_log, prec_log, label=f"Logit (PR-AUC={pr_auc:.3f})")
ax[1].plot(rec_dt, prec_dt, label=f"Arbre (PR-AUC={pr_auc_tree:.3f})")
ax[1].plot(rec_rf, prec_rf, label=f"RF (PR-AUC={pr_auc_rf:.3f})")
ax[1].set_title("Precision–Recall – Comparaison modèles")
ax[1].set_xlabel("Recall")
ax[1].set_ylabel("Precision")
ax[1].grid(True)
ax[1].legend()

plt.tight_layout()
plt.show()


## Modèles ensemblistes

In [ ]:
# ======================================
#   PREPARATION BASE UCI POUR CATBOOST
# ======================================

import pandas as pd
import numpy as np

# 1) Charger la base depuis UCI
from ucimlrepo import fetch_ucirepo 

dataset = fetch_ucirepo(id=296)

# X = features brutes
X = dataset.data.features.copy()

# Y = cible brute (readmitted)
Y = dataset.data.targets.copy()

# Fusion pour nettoyage plus simple
data = pd.concat([X, Y], axis=1)

print("Shape brut :", data.shape)
display(data.head())

# ======================================
#   2) Nettoyage minimal des valeurs manquantes
# ======================================

data = data.replace(["?", "NA", "na", "NaN", "nan", "None", ""], np.nan)

# ======================================
#   3) Conversion de la cible readmitted en binaire
# ======================================

# Certaines versions UCI utilisent ("<30", ">30", "NO")
if data["readmitted"].dtype == "object":
    data["readmitted"] = data["readmitted"].replace({
        "<30": 1,
        ">30": 1,
        "NO": 0
    })

# Vérification
print("Valeurs uniques de la cible :", data["readmitted"].unique())

# ======================================
#   4) Séparation X / y
# ======================================

X = data.drop("readmitted", axis=1)
y = data["readmitted"].astype(int)

# ======================================
#   5) Détection automatique des colonnes catégorielles
# ======================================

cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Nombre de colonnes catégorielles : {len(cat_features)}")
print("Aperçu des colonnes cat. :", cat_features[:20])

# ======================================
#   FIX : CatBoost n'accepte pas NaN dans les colonnes catégorielles
# ======================================

for col in cat_features:
    # convertir en string et remplacer les NaN
    X[col] = X[col].astype(str)
    X[col] = X[col].replace("nan", "Unknown")   # string "nan" que pandas produit

# ======================================
#   6) Split Train / Test
# ======================================

from sklearn.model_selection import train_test_split

X_train_cb, X_test_cb, y_train_cb, y_test_cb = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train :", X_train_cb.shape, "  Test :", X_test_cb.shape)

# ======================================
#   7) Vérification finale
# ======================================

print("\nTypes des variables dans le train:")
print(X_train_cb.dtypes)

print("\nAperçu du train :")
display(X_train_cb.head())


In [ ]:
# ======================================
#   CATBOOST : ENTRAINEMENT & EVALUATION
# ======================================

from catboost import CatBoostClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_recall_curve, classification_report, confusion_matrix, f1_score
)

# === 1. Entraînement CatBoost ===

cat_model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    class_weights=[1, 4],     # gère le déséquilibre (à ajuster si besoin)
    random_seed=42,
    verbose=100
)

cat_model.fit(
    X_train_cb, y_train_cb,
    cat_features=cat_features,
    eval_set=(X_test_cb, y_test_cb)
)

# === 2. Probabilités sur le test ===

proba_cat = cat_model.predict_proba(X_test_cb)[:, 1]

# === 3. Métriques probabilistes ===

roc_auc_cat = roc_auc_score(y_test_cb, proba_cat)
pr_auc_cat  = average_precision_score(y_test_cb, proba_cat)
brier_cat   = brier_score_loss(y_test_cb, proba_cat)

print("\n=== Performances CATBOOST sur test ===")
print(f"ROC-AUC : {roc_auc_cat:.4f}")
print(f"PR-AUC  : {pr_auc_cat:.4f}")
print(f"Brier   : {brier_cat:.5f}")

# === 4. Seuil optimal F1 ===

prec_cat, rec_cat, thr_cat = precision_recall_curve(y_test_cb, proba_cat)
f1_curve_cat = 2 * prec_cat * rec_cat / (prec_cat + rec_cat + 1e-12)

best_idx_cat = int(f1_curve_cat.argmax())
best_thr_cat = thr_cat[max(best_idx_cat - 1, 0)] if len(thr_cat) > 0 else 0.5

print(f"\nSeuil optimal CATBOOST (F1 max) : {best_thr_cat:.4f}")

# Prédiction binaire
y_pred_cat = (proba_cat >= best_thr_cat).astype(int)

# === 5. F1 pos / F1 neg / Harmonic F1 ===

f1_pos_cat = f1_score(y_test_cb, y_pred_cat)
f1_neg_cat = f1_score(1 - y_test_cb, 1 - y_pred_cat)
harm_cat   = (2 * f1_pos_cat * f1_neg_cat) / (f1_pos_cat + f1_neg_cat + 1e-12)

print("\n=== F1 Scores CATBOOST ===")
print(f"F1_POS      : {f1_pos_cat:.4f}")
print(f"F1_NEG      : {f1_neg_cat:.4f}")
print(f"Harmonic F1 : {harm_cat:.4f}")

# === 6. Classification report ===

print("\n=== Rapport de classification CATBOOST (test) ===")
print(classification_report(y_test_cb, y_pred_cat, digits=3))

# === 7. Matrice de confusion ===

cm_cat = confusion_matrix(y_test_cb, y_pred_cat)
print("Matrice de confusion (test):\n", cm_cat)


In [ ]:
# ======================================
#   PLOTS CATBOOST : ROC + PR CURVES
# ======================================

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

plt.figure(figsize=(14,5))

# -------------------------------------------------------
# 1) ROC curve
# -------------------------------------------------------
plt.subplot(1,2,1)
fpr_cat, tpr_cat, _ = roc_curve(y_test_cb, proba_cat)

plt.plot(fpr_cat, tpr_cat, label=f"CatBoost (AUC={roc_auc_cat:.3f})")
plt.plot([0,1], [0,1], "k--", alpha=0.5)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — CatBoost")
plt.grid(True)
plt.legend()

# -------------------------------------------------------
# 2) Precision–Recall curve
# -------------------------------------------------------
plt.subplot(1,2,2)
plt.plot(rec_cat, prec_cat, label=f"CatBoost (PR-AUC={pr_auc_cat:.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve — CatBoost")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()


## adaboost


In [ ]:
X = pd.read_csv("data/X_final.csv")
y = pd.read_csv("data/y.csv").squeeze()
y.name = "target"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# diag_1, diag_2, diag_3 sont des codes ICD-9. On les regroupe en catégories médicales.

from sklearn.preprocessing import OrdinalEncoder

def map_icd9(code):
    if code.startswith('V') or code.startswith('E'):
        return 'Other'
    try:
        c = float(code)
    except:
        return 'Other' 
    if 390 <= c <= 459 or c == 785: return 'Circulatory'
    if 460 <= c <= 519 or c == 786: return 'Respiratory'
    if 520 <= c <= 579 or c == 787: return 'Digestive'
    if str(int(c)).startswith('250'): return 'Diabetes'
    if 800 <= c <= 999: return 'Injury'
    if 710 <= c <= 739: return 'Musculoskeletal'
    if 580 <= c <= 629 or c == 788: return 'Genitourinary'
    if 140 <= c <= 239: return 'Neoplasms'
    return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    X[col] = X[col].astype(str).apply(map_icd9)


# Identifie les colonnes non numériques (normalement diag_1, diag_2, diag_3)
non_numeric_cols = X.select_dtypes(exclude=['number']).columns
print("Colonnes non numériques :", list(non_numeric_cols))

# Encodage ordinal
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[non_numeric_cols] = encoder.fit_transform(X[non_numeric_cols])

# Vérification
print(X.dtypes.value_counts())


# Split stratifié train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_recall_curve, classification_report, confusion_matrix, f1_score
)


# ======================================
#        ADABOOST : ENTRAINEMENT & TEST
# ======================================

from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_recall_curve, classification_report, confusion_matrix, f1_score
)

# === 1. Modèle AdaBoost ===
# Stump par défaut, on peut mettre un arbre plus profond si besoin.

ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=2),
    n_estimators=300,
    learning_rate=0.5,
    random_state=42
)


ada.fit(X_train, y_train)

# === 2. Probabilités sur test ===
proba_ada = ada.predict_proba(X_test)[:, 1]

# === 3. Métriques probabilistes ===
roc_auc_ada = roc_auc_score(y_test, proba_ada)
pr_auc_ada  = average_precision_score(y_test, proba_ada)
brier_ada   = brier_score_loss(y_test, proba_ada)

print("\n=== Performances ADABOOST sur test ===")
print(f"ROC-AUC : {roc_auc_ada:.4f}")
print(f"PR-AUC  : {pr_auc_ada:.4f}")
print(f"Brier   : {brier_ada:.5f}")

# === 4. Seuil optimal F1 ===
prec_ada, rec_ada, thr_ada = precision_recall_curve(y_test, proba_ada)
f1_curve_ada = 2 * prec_ada * rec_ada / (prec_ada + rec_ada + 1e-12)

best_idx_ada = int(f1_curve_ada.argmax())
best_thr_ada = thr_ada[max(best_idx_ada - 1, 0)] if len(thr_ada) > 0 else 0.5

print(f"\nSeuil optimal ADABOOST (F1 max) : {best_thr_ada:.4f}")

# Prédiction binaire
y_pred_ada = (proba_ada >= best_thr_ada).astype(int)

# === 5. F1 Scores ===
f1_pos_ada = f1_score(y_test, y_pred_ada)
f1_neg_ada = f1_score(1 - y_test, 1 - y_pred_ada)
harm_ada   = (2 * f1_pos_ada * f1_neg_ada) / (f1_pos_ada + f1_neg_ada + 1e-12)

print("\n=== F1 Scores ADABOOST ===")
print(f"F1_POS      : {f1_pos_ada:.4f}")
print(f"F1_NEG      : {f1_neg_ada:.4f}")
print(f"Harmonic F1 : {harm_ada:.4f}")

# === 6. Rapport de classification ===
print("\n=== Rapport de classification ADABOOST (test) ===")
print(classification_report(y_test, y_pred_ada, digits=3))

# === 7. Matrice de confusion ===
cm_ada = confusion_matrix(y_test, y_pred_ada)
print("Matrice de confusion (test):\n", cm_ada)


In [ ]:
# ======================================
#   ROC & PR COMPARATIFS : ADABOOST vs CATBOOST
# ======================================

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve

# --- Courbes ROC ---
fpr_cat, tpr_cat, _ = roc_curve(y_test_cb, proba_cat)
fpr_ada, tpr_ada, _ = roc_curve(y_test, proba_ada)

# --- Courbes PR ---
prec_cb_plot, rec_cb_plot, _ = precision_recall_curve(y_test_cb, proba_cat)
prec_ada_plot, rec_ada_plot, _ = precision_recall_curve(y_test, proba_ada)

plt.figure(figsize=(15,6))

# ----------------------------------------------------
# 1) ROC COMPARATIF
# ----------------------------------------------------
plt.subplot(1,2,1)

plt.plot(fpr_cat, tpr_cat, label=f"CatBoost (AUC={roc_auc_cat:.3f})", linewidth=2)
plt.plot(fpr_ada, tpr_ada, label=f"AdaBoost (AUC={roc_auc_ada:.3f})", linewidth=2)

plt.plot([0,1],[0,1], "k--", alpha=0.5)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — CatBoost vs AdaBoost")
plt.grid(True)
plt.legend()


# ----------------------------------------------------
# 2) PRECISION–RECALL COMPARATIF
# ----------------------------------------------------
plt.subplot(1,2,2)

plt.plot(rec_cb_plot, prec_cb_plot, 
         label=f"CatBoost (PR-AUC={pr_auc_cat:.3f})", linewidth=2)
plt.plot(rec_ada_plot, prec_ada_plot, 
         label=f"AdaBoost (PR-AUC={pr_auc_ada:.3f})", linewidth=2)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall — CatBoost vs AdaBoost")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()


## XG Boost

In [ ]:
# ======================================
#   XGBOOST : ENTRAINEMENT & EVALUATION
# ======================================

from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_recall_curve, classification_report, confusion_matrix, f1_score
)

# === 1. Modèle XGBoost ===
xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    gamma=0,
    objective="binary:logistic",
    eval_metric="logloss",     # important pour éviter warning
    tree_method="hist",        # RAPIDE
    random_state=42
)
# Nettoyage des noms de colonnes pour XGBoost
X_train.columns = (
    X_train.columns.astype(str)
        .str.replace('[', '(', regex=False)
        .str.replace(']', ')', regex=False)
        .str.replace('<', 'inf_', regex=False)
        .str.replace('>', 'sup_', regex=False)
        .str.replace(',', '_', regex=False)
        .str.replace(' ', '_', regex=False)
)

X_test.columns = X_train.columns  # aligner les deux


xgb.fit(X_train, y_train)

# === 2. Probabilités ===
proba_xgb = xgb.predict_proba(X_test)[:, 1]

# === 3. Métriques probabilistes ===
roc_auc_xgb = roc_auc_score(y_test, proba_xgb)
pr_auc_xgb  = average_precision_score(y_test, proba_xgb)
brier_xgb   = brier_score_loss(y_test, proba_xgb)

print("\n=== Performances XGBOOST sur test ===")
print(f"ROC-AUC : {roc_auc_xgb:.4f}")
print(f"PR-AUC  : {pr_auc_xgb:.4f}")
print(f"Brier   : {brier_xgb:.5f}")

# === 4. Seuil optimal F1 ===
prec_xgb, rec_xgb, thr_xgb = precision_recall_curve(y_test, proba_xgb)
f1_curve_xgb = 2 * prec_xgb * rec_xgb / (prec_xgb + rec_xgb + 1e-12)

best_idx_xgb = int(f1_curve_xgb.argmax())
best_thr_xgb = thr_xgb[max(best_idx_xgb - 1, 0)] if len(thr_xgb) > 0 else 0.5

print(f"\nSeuil optimal XGBOOST (F1 max) : {best_thr_xgb:.4f}")

# Prédiction binaire
y_pred_xgb = (proba_xgb >= best_thr_xgb).astype(int)

# === 5. F1 Scores ===
f1_pos_xgb = f1_score(y_test, y_pred_xgb)
f1_neg_xgb = f1_score(1 - y_test, 1 - y_pred_xgb)
harm_xgb   = (2 * f1_pos_xgb * f1_neg_xgb) / (f1_pos_xgb + f1_neg_xgb + 1e-12)

print("\n=== F1 Scores XGBOOST ===")
print(f"F1_POS      : {f1_pos_xgb:.4f}")
print(f"F1_NEG      : {f1_neg_xgb:.4f}")
print(f"Harmonic F1 : {harm_xgb:.4f}")

# === 6. Classification report ===
print("\n=== Rapport de classification XGBOOST ===")
print(classification_report(y_test, y_pred_xgb, digits=3))

# === 7. Matrice de confusion ===
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
print("Matrice de confusion (test):\n", cm_xgb)


In [ ]:
# CatBoost
y_true_cat = y_test_cb
# AdaBoost et XGBoost
y_true_ax  = y_test

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve

plt.figure(figsize=(15,6))

# ============================
#       ROC CURVES
# ============================
plt.subplot(1,2,1)

# AdaBoost
fpr_ada, tpr_ada, _ = roc_curve(y_true_ax, proba_ada)
plt.plot(fpr_ada, tpr_ada, label=f"AdaBoost (AUC={roc_auc_ada:.3f})", linewidth=2)

# CatBoost
fpr_cat, tpr_cat, _ = roc_curve(y_true_cat, proba_cat)
plt.plot(fpr_cat, tpr_cat, label=f"CatBoost (AUC={roc_auc_cat:.3f})", linewidth=2)

# XGBoost
fpr_xgb, tpr_xgb, _ = roc_curve(y_true_ax, proba_xgb)
plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC={roc_auc_xgb:.3f})", linewidth=2)

# Diagonale
plt.plot([0,1],[0,1],"k--", alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC — AdaBoost vs CatBoost vs XGBoost")
plt.grid(True)
plt.legend()


# ============================
#   PRECISION–RECALL CURVES
# ============================
plt.subplot(1,2,2)

# AdaBoost
prec_ada_plot, rec_ada_plot, _ = precision_recall_curve(y_true_ax, proba_ada)
plt.plot(rec_ada_plot, prec_ada_plot,
         label=f"AdaBoost (PR-AUC={pr_auc_ada:.3f})", linewidth=2)

# CatBoost
prec_cat_plot, rec_cat_plot, _ = precision_recall_curve(y_true_cat, proba_cat)
plt.plot(rec_cat_plot, prec_cat_plot,
         label=f"CatBoost (PR-AUC={pr_auc_cat:.3f})", linewidth=2)

# XGBoost
prec_xgb_plot, rec_xgb_plot, _ = precision_recall_curve(y_true_ax, proba_xgb)
plt.plot(rec_xgb_plot, prec_xgb_plot,
         label=f"XGBoost (PR-AUC={pr_auc_xgb:.3f})", linewidth=2)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall — AdaBoost vs CatBoost vs XGBoost")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
import numpy as np

# Importance brute
importance = cat_model.get_feature_importance()

# Création d’un tableau propre
feat_imp = pd.DataFrame({
    "Feature": X_train_cb.columns,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

print("\n=== Feature Importance CatBoost ===")
display(feat_imp.head(20))


## Réseaux de neurones

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, f1_score,
    classification_report, confusion_matrix
)

RANDOM_STATE = 42

# === Chargement des features sélectionnées ===
X = pd.read_csv("data/X_selections.csv")
y = pd.read_csv("data/y.csv").squeeze().astype(int)
y.name = "target"

print(f"Shape X: {X.shape} | Shape y: {y.shape}")
print("Distribution de la cible:\n", y.value_counts(normalize=True).round(3))

# Split stratifié
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "| Test:", X_test.shape)


In [ ]:
scaler = StandardScaler()

mlp_small = Pipeline([
    ("scaler", scaler),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(64,),
        activation="relu",
        alpha=1e-4,
        batch_size=512,
        learning_rate_init=1e-3,
        max_iter=100,
        early_stopping=True,
        random_state=RANDOM_STATE
    ))
])

mlp_medium = Pipeline([
    ("scaler", scaler),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        alpha=1e-4,
        batch_size=512,
        learning_rate_init=1e-3,
        max_iter=100,
        early_stopping=True,
        random_state=RANDOM_STATE
    ))
])

mlp_large = Pipeline([
    ("scaler", scaler),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(256, 128, 64),
        activation="relu",
        alpha=1e-4,
        batch_size=512,
        learning_rate_init=1e-3,
        max_iter=120,
        early_stopping=True,
        random_state=RANDOM_STATE
    ))
])


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"pr_auc": "average_precision", "roc_auc": "roc_auc"}

def cv_scores(pipe, name):
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    print(f"{name:>12} | PR-AUC: {scores['test_pr_auc'].mean():.3f} ± {scores['test_pr_auc'].std():.3f} "
          f"| ROC-AUC: {scores['test_roc_auc'].mean():.3f} ± {scores['test_roc_auc'].std():.3f}")
    return scores

scores_small  = cv_scores(mlp_small,  "MLP small")
scores_medium = cv_scores(mlp_medium, "MLP medium")
scores_large  = cv_scores(mlp_large,  "MLP large")


In [ ]:
mean_pr = {
    "small":  scores_small["test_pr_auc"].mean(),
    "medium": scores_medium["test_pr_auc"].mean(),
    "large":  scores_large["test_pr_auc"].mean()
}
best_name = max(mean_pr, key=mean_pr.get)
print("\n=> Meilleur MLP (PR-AUC CV) :", best_name)

if best_name == "small":
    best_mlp = mlp_small
elif best_name == "medium":
    best_mlp = mlp_medium
else:
    best_mlp = mlp_large


In [ ]:
# Entraînement sur tout le train
best_mlp.fit(X_train, y_train)

# Probabilités sur test
proba_test = best_mlp.predict_proba(X_test)[:, 1]

print(f"Test PR-AUC  : {average_precision_score(y_test, proba_test):.3f}")
print(f"Test ROC-AUC : {roc_auc_score(y_test, proba_test):.3f}")


In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, proba_test)
f1 = 2 * precision * recall / (precision + recall + 1e-12)
idx = np.nanargmax(f1)
thr = thresholds[idx-1] if 0 < idx < len(thresholds) else 0.5

y_pred = (proba_test >= thr).astype(int)

print(f"Seuil choisi (max F1) : {thr:.3f} | F1 = {f1[idx]:.3f}\n")
print("Classification report (MLP):")
print(classification_report(y_test, y_pred, digits=3))

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve, auc

# Courbe PR
precision, recall, _ = precision_recall_curve(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)

plt.figure(figsize=(6,5))
plt.plot(recall, precision, label=f"MLP (PR-AUC = {pr_auc:.3f})")
plt.axhline(y=y_test.mean(), ls="--", color="grey", label="Baseline (prévalence)")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Courbe Precision–Recall (MLP)")
plt.legend()
plt.tight_layout()
plt.show()

# Courbe ROC
fpr, tpr, _ = roc_curve(y_test, proba_test)
roc_auc = roc_auc_score(y_test, proba_test)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"MLP (ROC-AUC = {roc_auc:.3f})")
plt.plot([0,1],[0,1],"k--", label="Aléatoire")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("Courbe ROC (MLP)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
plt.hist(proba_test[y_test==0], bins=30, alpha=0.6, label="Classe 0")
plt.hist(proba_test[y_test==1], bins=30, alpha=0.6, label="Classe 1")
plt.axvline(x=thr, color="red", linestyle="--", label=f"Seuil = {thr:.3f}")
plt.xlabel("Probabilité prédite (classe 1)")
plt.ylabel("Nombre d'observations")
plt.title("Distribution des probabilités (MLP)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Accès au modèle interne
mlp_clf = best_mlp.named_steps["clf"]

plt.figure(figsize=(6,4))
plt.plot(mlp_clf.loss_curve_)
plt.xlabel("Itérations")
plt.ylabel("Loss")
plt.title("Courbe de loss (MLP)")
plt.tight_layout()
plt.show()

### X final

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, f1_score,
    classification_report, confusion_matrix
)

RANDOM_STATE = 42

# === 1. Chargement base complète ===
X_full = pd.read_csv("data/X_final.csv")
y = pd.read_csv("data/y.csv").squeeze().astype(int)
y.name = "target"

print(f"Shape X_full: {X_full.shape} | Shape y: {y.shape}")
print("Distribution de la cible:\n", y.value_counts(normalize=True).round(3))

# Split stratifié
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_full, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train full:", X_train_f.shape, "| Test full:", X_test_f.shape)

# === 2. Colonnes numériques / catégorielles ===
num_cols = X_full.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_full.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Nb num: {len(num_cols)} | Nb cat: {len(cat_cols)}")

# === 3. Préprocesseur pour toutes les variables ===
preproc_full = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ],
    remainder="drop"
)

In [ ]:
# === 3. Préprocesseur pour toutes les variables ===
preproc_full = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ],
    remainder="drop"
)

# === 4. MLP sur toutes les variables ===
mlp_full = Pipeline([
    ("preproc", preproc_full),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(256, 128, 64),
        activation="relu",
        alpha=1e-4,
        batch_size=512,
        learning_rate_init=1e-3,
        max_iter=120,
        early_stopping=True,
        random_state=RANDOM_STATE
    ))
])

# === 5. Validation croisée PR-AUC / ROC-AUC ===
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"pr_auc": "average_precision", "roc_auc": "roc_auc"}

scores_full = cross_validate(
    mlp_full,
    X_train_f, y_train_f,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

In [ ]:
print(f"MLP full | CV PR-AUC  : {scores_full['test_pr_auc'].mean():.3f} ± {scores_full['test_pr_auc'].std():.3f}")
print(f"MLP full | CV ROC-AUC : {scores_full['test_roc_auc'].mean():.3f} ± {scores_full['test_roc_auc'].std():.3f}")

# === 6. Entraînement final + test ===
mlp_full.fit(X_train_f, y_train_f)
proba_full = mlp_full.predict_proba(X_test_f)[:, 1]

print(f"\nTest PR-AUC  (MLP full): {average_precision_score(y_test_f, proba_full):.3f}")
print(f"Test ROC-AUC (MLP full): {roc_auc_score(y_test_f, proba_full):.3f}")


In [ ]:
# === 7. Seuil optimal (max F1) ===
precision, recall, thresholds = precision_recall_curve(y_test_f, proba_full)
f1 = 2 * precision * recall / (precision + recall + 1e-12)
idx = np.nanargmax(f1)
thr_full = thresholds[idx-1] if 0 < idx < len(thresholds) else 0.5

y_pred_full = (proba_full >= thr_full).astype(int)

print(f"\nSeuil choisi (max F1, MLP full) : {thr_full:.3f} | F1 = {f1[idx]:.3f}\n")
print("Classification report (MLP full) :")
print(classification_report(y_test_f, y_pred_full, digits=3))

cm_full = confusion_matrix(y_test_f, y_pred_full)
print("Confusion matrix (MLP full):\n", cm_full)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import loguniform

pipe_mlp = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(
        activation="relu",
        early_stopping=True,
        max_iter=150,
        random_state=42))
])

param_dist = {
    "clf__hidden_layer_sizes": [(64,), (128,), (128,64), (256,128,64)],
    "clf__alpha": loguniform(1e-5, 1e-2),
    "clf__batch_size": [256, 512, 1024],
    "clf__learning_rate_init": loguniform(1e-4, 5e-3),
}

search = RandomizedSearchCV(
    pipe_mlp,
    param_distributions=param_dist,
    n_iter=20,
    scoring="average_precision",
    cv=5,
    n_jobs=-1,
    random_state=42,
)

search.fit(X_train, y_train)
print("Best PR-AUC (CV):", search.best_score_)
print("Best params:", search.best_params_)


In [ ]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

ros = RandomOverSampler(random_state=42)

mlp_bal = ImbPipeline([
    ("ros", ros),
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(128,64),
        activation="relu",
        early_stopping=True,
        max_iter=150,
        random_state=42))
])


In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    best_mlp, X_test, y_test,
    n_repeats=10,
    scoring="average_precision",
    random_state=42,
    n_jobs=-1
)

imp = pd.Series(result.importances_mean, index=X.columns).sort_values(ascending=False)
print(imp.head(15))
